# Notebook 09 — Validation corrélation OCG → TCG

**Hypothèse** : les archetypes qui dominent l'OCG (Japon/Asie) dominent le TCG (Europe/Amérique) 3 à 6 mois après.

Si cette corrélation est validée, le signal OCG devient notre edge principal pour anticiper la méta TCG — et la valeur produit clé pour les boutiques.

**Plan :**
- Part A : Explorer les données OCG disponibles
- Part B : Calculer les meta_scores OCG par archetype/mois
- Part C : Backtester la corrélation OCG→TCG sur différents lags (2 à 7 mois)
- Part D : Visualiser les cas concrets
- Part E : Conclusion & signal boutiques

**Prérequis** : avoir lancé `python scripts/fetch_ocg_decks.py` au moins une fois.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os

DB = os.path.join(os.getcwd(), '..', 'data', 'yugioh.db')
conn = sqlite3.connect(DB)

ocg_n = pd.read_sql('SELECT COUNT(*) as n FROM tournament_decks WHERE ocg=1', conn).iloc[0,0]
tcg_n = pd.read_sql('SELECT COUNT(*) as n FROM tournament_decks WHERE ocg=0', conn).iloc[0,0]
print(f'Decklists OCG : {ocg_n}')
print(f'Decklists TCG : {tcg_n}')

if ocg_n == 0:
    print()
    print('⚠️  Aucune donnée OCG — lance d\'abord :')
    print('   python scripts/fetch_ocg_decks.py')
else:
    df = pd.read_sql("""
        SELECT strftime('%Y-%m', created) as month, archetype, COUNT(*) as n
        FROM tournament_decks
        WHERE ocg=1 AND archetype IS NOT NULL
        GROUP BY month, archetype ORDER BY month, n DESC
    """, conn)
    print(f'Mois OCG disponibles : {df["month"].nunique()}')
    print(f'Archetypes OCG uniques : {df["archetype"].nunique()}')
    print()
    print('Top 15 archetypes OCG :')
    print(df.groupby('archetype')['n'].sum().sort_values(ascending=False).head(15))

## Part B — Meta score OCG par archetype/mois

On utilise la même formule que le TCG : `meta_score = sqrt(share × placement_score_norm)`

In [ ]:
ocg_raw = pd.read_sql("""
    SELECT strftime('%Y-%m', created) as month, archetype,
           COUNT(*) as n_decks,
           AVG(CASE WHEN placement IS NOT NULL THEN placement END) as avg_placement
    FROM tournament_decks
    WHERE ocg=1 AND archetype IS NOT NULL AND illegal=0
    GROUP BY month, archetype
""", conn)

ocg_raw['month'] = pd.to_datetime(ocg_raw['month'])

monthly_total = ocg_raw.groupby('month')['n_decks'].sum().rename('total')
ocg_raw = ocg_raw.join(monthly_total, on='month')
ocg_raw['share'] = ocg_raw['n_decks'] / ocg_raw['total']

ocg_raw['placement_score'] = 1 / ocg_raw['avg_placement'].replace(0, np.nan)
monthly_max = ocg_raw.groupby('month')['placement_score'].max().rename('max_ps')
ocg_raw = ocg_raw.join(monthly_max, on='month')
ocg_raw['placement_score_norm'] = ocg_raw['placement_score'] / ocg_raw['max_ps']
ocg_raw['meta_score_ocg'] = np.sqrt(ocg_raw['share'] * ocg_raw['placement_score_norm'].fillna(ocg_raw['share']))

ocg_ms = ocg_raw[['month', 'archetype', 'meta_score_ocg', 'share', 'n_decks']].copy()
print(f'OCG meta_scores : {len(ocg_ms)} lignes')
print()
print('Top archetypes OCG par meta_score moyen :')
print(ocg_ms.groupby('archetype')['meta_score_ocg'].mean().sort_values(ascending=False).head(15))

## Part C — Backtest corrélation (lags 2 à 7 mois)

On teste : est-ce que `meta_score_ocg(T)` corrèle avec `meta_score_tcg(T + lag)` ?

Le lag qui maximise la corrélation = notre fenêtre d'avance sur la méta.

In [ ]:
from dateutil.relativedelta import relativedelta

tcg_ms = pd.read_sql('SELECT month, archetype, meta_score FROM meta_scores', conn, parse_dates=['month'])
tcg_ms.columns = ['month', 'archetype', 'meta_score_tcg']

ocg_archs = set(ocg_ms['archetype'].unique())
tcg_archs = set(tcg_ms['archetype'].unique())
common = ocg_archs & tcg_archs
print(f'Archetypes OCG  : {len(ocg_archs)}')
print(f'Archetypes TCG  : {len(tcg_archs)}')
print(f'Archetypes communs : {len(common)}')
print()

results = []
for lag in range(2, 8):
    shifted = ocg_ms.copy()
    shifted['month_tcg'] = shifted['month'].apply(lambda d: d + relativedelta(months=lag))
    merged = shifted.merge(
        tcg_ms,
        left_on=['month_tcg', 'archetype'],
        right_on=['month', 'archetype'],
        how='inner'
    )
    if len(merged) < 10:
        print(f'Lag {lag}m : pas assez de paires ({len(merged)})')
        continue
    corr, pval = pearsonr(merged['meta_score_ocg'], merged['meta_score_tcg'])
    sig = '✅' if pval < 0.05 else '❌'
    results.append({'lag_mois': lag, 'n_paires': len(merged), 'correlation': round(corr, 3), 'p_value': round(pval, 4)})
    print(f'Lag {lag}m : r={corr:.3f}  p={pval:.4f}  n={len(merged)}  {sig}')

res_df = pd.DataFrame(results)
best_lag = int(res_df.loc[res_df['correlation'].idxmax(), 'lag_mois']) if len(res_df) else None
print(f'\n→ Lag optimal : {best_lag} mois')

In [ ]:
# Visualiser les corrélations par lag
if len(res_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['green' if p < 0.05 else 'gray' for p in res_df['p_value']]
    ax.bar(res_df['lag_mois'], res_df['correlation'], color=colors, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axhline(0.5, color='green', linewidth=0.8, linestyle='--', alpha=0.5, label='r=0.5 (seuil signal fort)')
    ax.set_xlabel('Lag (mois)')
    ax.set_ylabel('Corrélation r')
    ax.set_title('Corrélation OCG→TCG par lag (vert = p<0.05)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Part D — Visualisation des cas concrets

On superpose OCG et TCG pour les archetypes communs les plus forts, avec le signal OCG décalé du lag optimal.

In [ ]:
if best_lag and common:
    top_ocg = ocg_ms.groupby('archetype')['meta_score_ocg'].mean().sort_values(ascending=False)
    top_plot = [a for a in top_ocg.index if a in common][:4]

    fig, axes = plt.subplots(len(top_plot), 1, figsize=(12, 4 * len(top_plot)))
    if len(top_plot) == 1:
        axes = [axes]

    for ax, arch in zip(axes, top_plot):
        o = ocg_ms[ocg_ms['archetype'] == arch].set_index('month')['meta_score_ocg'].sort_index()
        t = tcg_ms[tcg_ms['archetype'] == arch].set_index('month')['meta_score_tcg'].sort_index()
        o_shifted = o.copy()
        o_shifted.index = o_shifted.index + pd.DateOffset(months=best_lag)

        ax.plot(o.index, o.values, 'b-o', ms=4, label='OCG (réel)', lw=1.5)
        ax.plot(t.index, t.values, 'r-o', ms=4, label='TCG (réel)', lw=1.5)
        ax.plot(o_shifted.index, o_shifted.values, 'b--', alpha=0.5, label=f'OCG décalé +{best_lag}m', lw=1)
        ax.set_title(arch, fontsize=12)
        ax.legend(fontsize=9)
        ax.set_ylabel('meta_score')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out = os.path.join(os.getcwd(), '..', 'data', 'ocg_tcg_correlation.png')
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Graphe sauvegardé : data/ocg_tcg_correlation.png')
else:
    print('Pas assez de données — lance fetch_ocg_decks.py')

## Part E — Conclusion & signal boutiques

In [ ]:
print('=' * 50)
print('RÉSUMÉ — Signal OCG→TCG')
print('=' * 50)

if len(results) > 0:
    best = res_df.loc[res_df['correlation'].idxmax()]
    print(f'Lag optimal    : {int(best["lag_mois"])} mois')
    print(f'Corrélation    : r = {best["correlation"]}')
    print(f'p-value        : {best["p_value"]}')
    print(f'Paires testées : {int(best["n_paires"])}')
    print()
    if best['correlation'] > 0.5 and best['p_value'] < 0.05:
        print('✅ Signal VALIDÉ — OCG est un prédicteur fiable du TCG')
        print(f'   → Les boutiques ont {int(best["lag_mois"])} mois d\'avance sur la méta TCG')
        print('   → Prochaine étape : SB-Z (score alerte boutiques)')
    elif best['correlation'] > 0.3:
        print('⚠️  Signal PARTIEL — corrélation modérée')
        print('   → Affiner sur des archetypes spécifiques')
    else:
        print('❌ Signal FAIBLE — corrélation insuffisante')
        print('   → Reconsidérer la stratégie boutiques')
else:
    print('⚠️  Pas de données OCG — lance fetch_ocg_decks.py')

conn.close()